# Model Deployment (Save/Load & API)
**Repositori**: Machine Learning
**Topik**: Menyimpan, memuat, dan mendeploy model ML
**Dataset**: Iris (built-in)
---
**Pendahuluan**: Model yang sudah dilatih perlu disimpan (serialisasi) agar bisa digunakan kembali tanpa training ulang. Joblib dan Pickle adalah dua library utama untuk serialisasi.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import pickle
import os

sns.set(style='whitegrid')


## 2. Train Model (Random Forest - Iris)


In [ ]:
iris = datasets.load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=target_names))


## 3. Menyimpan Model (Serialization)


In [ ]:
os.makedirs('../../models', exist_ok=True)
joblib.dump(model, '../../models/iris_rf_model.joblib')
joblib.dump(scaler, '../../models/iris_scaler.joblib')
print('Model saved with joblib!')
with open('../../models/iris_rf_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../../models/iris_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Model saved with pickle!')


## 4. Memuat Model (Deserialization)


In [ ]:
loaded_model = joblib.load('../../models/iris_rf_model.joblib')
loaded_scaler = joblib.load('../../models/iris_scaler.joblib')
print('Model loaded successfully!')
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
sample_scaled = loaded_scaler.transform(sample)
pred = loaded_model.predict(sample_scaled)
proba = loaded_model.predict_proba(sample_scaled)
print(f'Sample: {sample[0]}')
print(f'Predicted class: {target_names[pred[0]]}')
print(f'Probabilities: {dict(zip(target_names, proba[0]))}')


## 5. Batch Prediction


In [ ]:
samples = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.7, 3.0, 5.2, 2.3],
    [5.9, 3.0, 4.2, 1.5]
])
samples_scaled = loaded_scaler.transform(samples)
predictions = loaded_model.predict(samples_scaled)
probabilities = loaded_model.predict_proba(samples_scaled)
results = pd.DataFrame(samples, columns=feature_names)
results['Predicted'] = target_names[predictions]
results['Confidence'] = probabilities.max(axis=1)
print(results)


## 6. API Deployment dengan FastAPI (Kode Contoh)


Berikut adalah contoh kode untuk mendeploy model sebagai REST API menggunakan FastAPI:


In [ ]:
# Contoh: simpan sebagai `app.py`, jalankan dengan: uvicorn app:app --reload
# from fastapi import FastAPI
# from pydantic import BaseModel
# import joblib, numpy as np

# app = FastAPI(title='Iris Classifier API')
# model = joblib.load('../../models/iris_rf_model.joblib')
# scaler = joblib.load('../../models/iris_scaler.joblib')
# target_names = ['setosa', 'versicolor', 'virginica']

# class IrisFeatures(BaseModel):
#     sepal_length: float
#     sepal_width: float
#     petal_length: float
#     petal_width: float

# @app.get('/')
# def root():
#     return {'message': 'Iris Classifier API is running'}

# @app.post('/predict')
# def predict(features: IrisFeatures):
#     data = np.array([[features.sepal_length, features.sepal_width, features.petal_length, features.petal_width]])
#     scaled = scaler.transform(data)
#     pred = model.predict(scaled)[0]
#     proba = model.predict_proba(scaled)[0].tolist()
#     return {'prediction': int(pred), 'class_name': target_names[pred], 'probabilities': proba}

print('Contoh kode API FastAPI (lihat markdown di atas)')


## 7. Kesimpulan
Joblib adalah pilihan utama untuk serialisasi model scikit-learn karena lebih efisien untuk objek NumPy. Model yang sudah disimpan dapat diintegrasikan ke aplikasi web melalui REST API, memungkinkan prediksi real-time tanpa training ulang.
